In [2]:
import os
import glob
import pandas as pd

selected_files = [
    '/content/nba_player_stats_2026 2.csv',
    '/content/Advanced.csv',
    '/content/All-Star Selections.csv',
    '/content/Draft Pick History.csv',
    '/content/End of Season Teams (Voting).csv',
    '/content/End of Season Teams.csv',
    '/content/Opponent Stats Per 100 Poss.csv',
    '/content/Opponent Stats Per Game.csv',
    '/content/Opponent Totals.csv',
    '/content/Per 36 Minutes.csv',
    '/content/Per 100 Poss.csv',
    '/content/Player Award Shares.csv',
    '/content/Player Career Info.csv',
    '/content/Player Per Game.csv',
    '/content/Player Play By Play.csv',
    '/content/Player Season Info.csv',
    '/content/Player Shooting.csv',
    '/content/Player Totals.csv',
    '/content/Team Abbrev.csv',
    '/content/Team Stats Per 100 Poss.csv',
    '/content/Team Stats Per Game.csv',
    '/content/Team Summaries.csv',
    '/content/Team Totals.csv'
]


def resolve_csv_path(path):
    candidates = [path]
    if path.startswith('/content/'):
        candidates.append(path.replace('/content/', './'))
    if not path.startswith('/'):
        candidates.append(os.path.join(os.getcwd(), path))
    candidates.extend([
        os.path.join(os.getcwd(), os.path.basename(path)),
        os.path.join(os.getcwd(), 'data', os.path.basename(path)),
        os.path.join(os.getcwd(), 'datasets', os.path.basename(path)),
    ])
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return None


dfs = {}
resolved_files = []
for file_path in selected_files:
    resolved = resolve_csv_path(file_path)
    if resolved:
        resolved_files.append(resolved)
    else:
        name = os.path.basename(file_path)
        matches = glob.glob(f'**/{name}', recursive=True)
        if matches:
            resolved_files.append(matches[0])

for file_path in resolved_files:
    try:
        file_name = os.path.splitext(os.path.basename(file_path))[0]
        dfs[file_name] = pd.read_csv(file_path)
        print(f"Successfully loaded '{file_name}'")
    except Exception as e:
        print(f"Error loading '{file_path}': {e}")

if not dfs:
    print("No NBA CSV files were found; using a small built-in sample for archetype demo.")
    dfs['sample'] = pd.DataFrame({
        'Player': [
            'LeBron James', 'Stephen Curry', 'Nikola Jokic', 'Giannis Antetokounmpo',
            'Jayson Tatum', 'De\'Aaron Fox', 'Devin Booker', 'Rudy Gobert',
            'Anthony Davis', 'Luka Doncic'
        ],
        'PTS': [25.7, 26.4, 26.4, 30.4, 26.9, 26.6, 27.1, 14.0, 24.7, 33.9],
        'AST': [8.3, 6.3, 9.2, 5.9, 4.1, 5.6, 6.8, 1.1, 3.2, 9.8],
        'TRB': [7.3, 5.1, 12.8, 11.5, 8.1, 5.1, 4.5, 12.9, 12.3, 9.2],
        'STL': [1.3, 0.8, 1.3, 1.2, 1.0, 1.6, 1.1, 0.8, 2.2, 1.4],
        'BLK': [0.5, 0.4, 0.7, 1.2, 0.7, 0.2, 0.4, 2.1, 2.3, 0.5],
        'FG%': [0.50, 0.42, 0.58, 0.55, 0.47, 0.48, 0.45, 0.66, 0.53, 0.46],
        '3P%': [0.36, 0.42, 0.31, 0.30, 0.36, 0.36, 0.38, 0.00, 0.30, 0.35],
        'FT%': [0.76, 0.91, 0.82, 0.72, 0.85, 0.78, 0.86, 0.64, 0.82, 0.78],
        'USG%': [29.0, 27.0, 30.0, 35.0, 28.0, 31.0, 29.0, 16.0, 26.0, 35.0],
        'PER': [24.5, 23.7, 31.2, 30.8, 23.8, 24.1, 23.0, 25.3, 25.0, 33.5],
        'TS%': [0.58, 0.62, 0.64, 0.61, 0.57, 0.58, 0.58, 0.64, 0.59, 0.62],
        'VORP': [2.0, 2.1, 3.8, 3.5, 2.2, 1.7, 1.5, 2.7, 2.8, 4.0],
    })

No NBA CSV files were found; using a small built-in sample for archetype demo.


### Displaying the head of each loaded DataFrame

In [6]:
import pandas as pd
import numpy as np
import os

# Find the best available player-stats table from the loaded data.
def find_player_table(frame_dict):
    if not frame_dict:
        return None

    preferred_names = ['nba_player_stats_2026 2', 'Player Per Game', 'Player Totals', 'Per Game', 'Per 36 Minutes', 'Advanced', 'sample']
    for name in preferred_names:
        if name in frame_dict:
            return frame_dict[name]

    for df in frame_dict.values():
        cols = {str(c).strip().lower() for c in df.columns}
        if any(token in cols for token in {'player', 'player_name', 'playerid', 'name'}) and any(token in cols for token in {'pts', 'ast', 'trb', 'reb', 'fg%', 'fg_pct', 'points'}):
            return df

    return next(iter(frame_dict.values()), None)

stats_df = find_player_table(dfs)
per_game_df = dfs.get('Player Per Game')
advanced_df = dfs.get('Advanced')

if stats_df is None:
    raise ValueError('No usable player stats table was found for archetype classification.')


def clean_name(s):
    return str(s).strip() if pd.notna(s) else ''

# Standardize the main stats table.
main = stats_df.copy()
main.columns = [str(c).strip() for c in main.columns]

player_col = None
for candidate in ['PLAYER', 'player', 'Player', 'Name', 'PLAYER_NAME']:
    if candidate in main.columns:
        player_col = candidate
        break

if player_col is None:
    main['Player'] = [f'Player_{i + 1}' for i in range(len(main))]
else:
    main['Player'] = main[player_col].fillna('')
    main['Player'] = main['Player'].apply(clean_name)

main['Player'] = main['Player'].replace('', np.nan)
main['Player'] = main['Player'].fillna(pd.Series([f'Player_{i + 1}' for i in range(len(main))], index=main.index))

# Add common stat columns from the main file when available.
rename_map = {}
for col in main.columns:
    low = str(col).strip().lower()
    if low in {'pts', 'points', 'points_per_game', 'pts_per_game', 'pts_per_g', 'ppg'}:
        rename_map[col] = 'PTS'
    elif low in {'ast', 'assists', 'assists_per_game', 'ast_per_game', 'apg'}:
        rename_map[col] = 'AST'
    elif low in {'reb', 'trb', 'rebounds', 'rebounds_per_game', 'trb_per_game', 'rpg'}:
        rename_map[col] = 'TRB'
    elif low in {'stl', 'steals', 'steals_per_game', 'stl_per_game', 'spg'}:
        rename_map[col] = 'STL'
    elif low in {'blk', 'blocks', 'blocks_per_game', 'blk_per_game', 'bpg'}:
        rename_map[col] = 'BLK'
    elif low in {'fg_pct', 'fg%', 'field_goal_percentage', 'fg_percent'}:
        rename_map[col] = 'FG%'
    elif low in {'fg3_pct', '3p_pct', '3p%', 'fg3_percent', 'three_point_percentage'}:
        rename_map[col] = '3P%'
    elif low in {'ft_pct', 'ft%', 'free_throw_percentage'}:
        rename_map[col] = 'FT%'
    elif low in {'eff', 'player_efficiency_rating', 'per'}:
        rename_map[col] = 'PER'
    elif low in {'ast_tov', 'ast_tov_ratio'}:
        rename_map[col] = 'AST_TOV'
    elif low in {'stl_tov', 'stl_tov_ratio'}:
        rename_map[col] = 'STL_TOV'

main = main.rename(columns=rename_map)

# Merge in advanced metrics if available.
if advanced_df is not None:
    adv = advanced_df.copy()
    adv.columns = [str(c).strip() for c in adv.columns]
    adv['Player'] = adv.get('player', adv.get('PLAYER', adv.get('Player', pd.Series([''] * len(adv)))))
    adv['Player'] = adv['Player'].fillna('').apply(clean_name)
    adv_agg = (
        adv.groupby('Player', as_index=False)
        .agg(
            USG_pct=('usg_percent', 'mean'),
            VORP=('vorp', 'mean'),
            BPM=('bpm', 'mean'),
            WS=('ws', 'mean')
        )
    )
    main = main.merge(adv_agg, on='Player', how='left')

# Merge in per-game stats if available.
if per_game_df is not None and per_game_df is not stats_df:
    pg = per_game_df.copy()
    pg.columns = [str(c).strip() for c in pg.columns]
    pg['Player'] = pg.get('player', pg.get('PLAYER', pg.get('Player', pg.get('Name', pd.Series([''] * len(pg))))))
    pg['Player'] = pg['Player'].fillna('').apply(clean_name)
    pg_agg = {
        'PTS_pg': ('pts_per_game', 'mean') if 'pts_per_game' in pg.columns else ('pts', 'mean'),
        'AST_pg': ('ast_per_game', 'mean') if 'ast_per_game' in pg.columns else ('ast', 'mean'),
        'TRB_pg': ('trb_per_game', 'mean') if 'trb_per_game' in pg.columns else ('trb', 'mean'),
        'STL_pg': ('stl_per_game', 'mean') if 'stl_per_game' in pg.columns else ('stl', 'mean'),
        'BLK_pg': ('blk_per_game', 'mean') if 'blk_per_game' in pg.columns else ('blk', 'mean'),
        'FG_pct_pg': ('fg_percent', 'mean') if 'fg_percent' in pg.columns else ('fg_pct', 'mean'),
        'ThreeP_pct_pg': ('fg3_percent', 'mean') if 'fg3_percent' in pg.columns else ('3p_pct', 'mean'),
        'FT_pct_pg': ('ft_percent', 'mean') if 'ft_percent' in pg.columns else ('ft_pct', 'mean'),
    }
    pg_agg_df = pg.groupby('Player', as_index=False).agg(**pg_agg)
    main = main.merge(pg_agg_df, on='Player', how='left')

# Choose the most useful columns for scoring.
score_cols = ['PTS', 'AST', 'TRB', 'STL', 'BLK', 'FG%', '3P%', 'FT%', 'USG_pct', 'VORP', 'BPM', 'PER', 'AST_TOV', 'STL_TOV']
for col in score_cols:
    if col in main.columns:
        main[col] = pd.to_numeric(main[col], errors='coerce')

# Fill missing values with player-level defaults so the scoring still works.
main['PTS'] = main['PTS'].fillna(main.get('PTS_pg', pd.Series([np.nan] * len(main))).fillna(0))
main['AST'] = main['AST'].fillna(main.get('AST_pg', pd.Series([np.nan] * len(main))).fillna(0))
main['TRB'] = main['TRB'].fillna(main.get('TRB_pg', pd.Series([np.nan] * len(main))).fillna(0))
main['STL'] = main['STL'].fillna(main.get('STL_pg', pd.Series([np.nan] * len(main))).fillna(0))
main['BLK'] = main['BLK'].fillna(main.get('BLK_pg', pd.Series([np.nan] * len(main))).fillna(0))
main['FG%'] = main['FG%'].fillna(main.get('FG_pct_pg', pd.Series([np.nan] * len(main))).fillna(0.45))
main['3P%'] = main['3P%'].fillna(main.get('ThreeP_pct_pg', pd.Series([np.nan] * len(main))).fillna(0.35))
main['FT%'] = main['FT%'].fillna(main.get('FT_pct_pg', pd.Series([np.nan] * len(main))).fillna(0.75))
main['USG_pct'] = main.get('USG_pct', pd.Series([20] * len(main)))
main['USG_pct'] = pd.to_numeric(main['USG_pct'], errors='coerce').fillna(20)
main['VORP'] = main.get('VORP', pd.Series([0] * len(main)))
main['VORP'] = pd.to_numeric(main['VORP'], errors='coerce').fillna(0)
main['BPM'] = main.get('BPM', pd.Series([0] * len(main)))
main['BPM'] = pd.to_numeric(main['BPM'], errors='coerce').fillna(0)
main['PER'] = main.get('PER', pd.Series([15] * len(main)))
main['PER'] = pd.to_numeric(main['PER'], errors='coerce').fillna(15)
main['AST_TOV'] = main.get('AST_TOV', pd.Series([1.6] * len(main)))
main['AST_TOV'] = pd.to_numeric(main['AST_TOV'], errors='coerce').fillna(1.6)
main['STL_TOV'] = main.get('STL_TOV', pd.Series([0.4] * len(main)))
main['STL_TOV'] = pd.to_numeric(main['STL_TOV'], errors='coerce').fillna(0.4)

# Create scoring features.
def archetype_scores(row):
    pts = row.get('PTS', 0) or 0
    ast = row.get('AST', 0) or 0
    trb = row.get('TRB', 0) or 0
    stl = row.get('STL', 0) or 0
    blk = row.get('BLK', 0) or 0
    fg = row.get('FG%', 0) or 0
    tp = row.get('3P%', 0) or 0
    ft = row.get('FT%', 0) or 0
    usg = row.get('USG_pct', 0) or 0
    vorp = row.get('VORP', 0) or 0
    bpm = row.get('BPM', 0) or 0
    per = row.get('PER', 0) or 0
    ast_tov = row.get('AST_TOV', 0) or 0
    stl_tov = row.get('STL_TOV', 0) or 0

    return {
        'Playmaker': ast * 1.6 + usg * 0.8 + ast_tov * 1.4 + per * 0.4,
        'Scorer': pts * 1.4 + usg * 0.9 + tp * 2.4 + fg * 1.4,
        'Ball_Handler': ast * 1.2 + pts * 0.9 + tp * 2.0 + usg * 0.6,
        'Slasher': pts * 0.9 + trb * 0.8 + stl * 1.7 + fg * 1.3,
        'Three_Level_Shooter': tp * 4.0 + fg * 1.8 + ft * 2.0 + pts * 0.6,
        'Interior_Post': trb * 1.4 + blk * 2.2 + fg * 1.6 + vorp * 0.8,
        'Defensive_Anchor': blk * 2.5 + trb * 1.2 + stl * 1.8 + bpm * 0.6,
        'Versatile_Forward': trb * 1.2 + pts * 0.8 + ast * 0.7 + stl * 1.2,
        'Two_Way_Wing': stl * 2.0 + blk * 1.5 + pts * 0.8 + tp * 2.1,
        'Facilitator': ast * 2.0 + usg * 0.6 + ast_tov * 1.8 + bpm * 0.5,
    }

score_df = pd.DataFrame([archetype_scores(row) for _, row in main.iterrows()])
score_df.index = main.index

primary = []
secondary = []
for _, row in score_df.iterrows():
    ranked = row.sort_values(ascending=False)
    primary.append(ranked.index[0])
    secondary.append(ranked.index[1])

main['Primary_Archetype'] = primary
main['Secondary_Archetype'] = secondary

for archetype in score_df.columns:
    main[f'{archetype}_Score'] = score_df[archetype]

# Keep a clean output table.
out = main[['Player', 'Primary_Archetype', 'Secondary_Archetype'] + [c for c in main.columns if c.endswith('_Score')]].copy()
out = out.dropna(subset=['Player'])
out = out[out['Player'] != '']

out_path = os.path.join(os.getcwd(), 'player_archetypes.csv')
out.to_csv(out_path, index=False)

print('Archetype assignment complete.')
print(out.head(20))
print(f'Saved to {out_path}')
print('\nArchetype counts:')
print(out['Primary_Archetype'].value_counts().to_string())

Archetype assignment complete.
                  Player Primary_Archetype Secondary_Archetype  \
0           LeBron James            Scorer        Ball_Handler   
1          Stephen Curry            Scorer        Ball_Handler   
2           Nikola Jokic            Scorer        Ball_Handler   
3  Giannis Antetokounmpo            Scorer        Ball_Handler   
4           Jayson Tatum            Scorer        Ball_Handler   
5           De'Aaron Fox            Scorer        Ball_Handler   
6           Devin Booker            Scorer        Ball_Handler   
7            Rudy Gobert            Scorer           Playmaker   
8          Anthony Davis            Scorer   Versatile_Forward   
9            Luka Doncic            Scorer        Ball_Handler   

   Playmaker_Score  Scorer_Score  Ball_Handler_Score  Slasher_Score  \
0            41.32        55.544               45.81         31.830   
1            37.80        56.556               44.16         29.746   
2            45.44        56.